COSC 301 Project
Group 4
Instacart Market Basket Analysis

**Importing Files**

The way to run this program is by running bash scripts/get_data.sh in terminal 

In [7]:
import pandas as pd               # for data manipulation
import matplotlib.pyplot as plt   # for plotting 
import seaborn as sns             # for statistical graph

In [12]:
orders = pd.read_csv('../raw_data/orders.csv' )
products = pd.read_csv('../raw_data/products.csv')
order_products_prior = pd.read_csv('../raw_data/order_products__prior.csv')
aisles = pd.read_csv('../raw_data/aisles.csv')
order_products_train = pd.read_csv('../raw_data/order_products__train.csv')

In [9]:
orders.head()

,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,2539329,1,prior,1,2,8,NaN
1,2398795,1,prior,2,3,7,15.0
2,473747,1,prior,3,3,12,21.0
3,2254736,1,prior,4,4,7,29.0
4,431534,1,prior,5,4,15,28.0


# Aadil's code

In [ ]:
prior_orders = pd.merge(orders, order_products_prior, on='order_id', how='inner')
print(prior_orders.memory_usage(deep=True).sum() / (1024**3))
print(len(orders))
print(len(products))
print(len(order_products_prior))
print(len(aisles))
print(f"prior_orders: {len(prior_orders)} rows")

## Cleaning data

How to clean drop columns remove na values or rows or columns maybe check how many na values in a row or column and only remove if greater than 5 or 10% or maybe remove all rows that have an na in them

In [ ]:
#code for removing na 
prior_orders_cleaned = prior_orders.dropna()

print(f"prior_orders_cleaned: {len(prior_orders_cleaned)} rows")

## Research Question 1

How accurately will we recommend the top 5 products a user is most likely to purchase in their next order based on their previous orders (using the order_product_prior csv file)?

Steps to answer the question

Step 1: create dataframe with all except last order of each user. use a loop to create dataframe for each user.

Step 2: Calculate top 5 products based on frequency of ordering the product that the user has order in all of their order excluding their latest order OR Use ML algorithm to recommend 5 products based on what user has previously ordered OR BOTH

Step 3: recommend top 5 products a user will order in the next order based on previous orders. compute accuracy by comparing the recommended 5 products with the actual orders ordered in the last order of the user

In [ ]:
#step 1

#making the connection to a created database
connection = sqlite3.connect("instacart.db")

#sending dataframe to the sql database
prior_orders.to_sql("prior_orders", connection, if_exists = "replace", index = False)

query = """
        SELECT order_id, user_id, product_id, order_number
        FROM (
            SELECT 
                order_id,
                user_id,
                product_id,
                order_number,
                MAX(order_number) OVER (PARTITION BY user_id) AS max_order_number
            FROM prior_orders 
        )
        WHERE order_number < max_order_number 
    """
    
df_without_last_orders_of_users = pd.read_sql(query, connection)

df_without_last_orders_of_users.head()

In [ ]:
product_counts = (
                df_without_last_orders_of_users
                .groupby(["user_id", "product_id"])
                .size()
                .reset_index(name = "order_count")
)

product_counts["rank"] = (
                        product_counts
                        .groupby("user_id")["order_count"]
                        .rank(method = "first", ascending = False)
)

top5_products = product_counts[product_counts['rank'] <= 5]

top5_products.head(10)

# Continuation of Raunak's code

In [13]:
# Join prior order-products with orders to get user_id + order_number
op_prior = order_products_prior.merge(
    orders[['order_id', 'user_id', 'order_number']],
    on='order_id',
    how='left'
)

# add product meta (name/aisle/department)
prod_meta = products[['product_id', 'product_name', 'aisle_id', 'department_id']].copy()
op_prior = op_prior.merge(prod_meta, on='product_id', how='left')

op_prior.head()

,order_id,product_id,add_to_cart_order,reordered,user_id,order_number,product_name,aisle_id,department_id
0,2,33120,1,1,202279,3,Organic Egg Whites,86,16
1,2,28985,2,1,202279,3,Michigan Organic Kale,83,4
2,2,9327,3,0,202279,3,Garlic Powder,104,13
3,2,45918,4,1,202279,3,Coconut Butter,19,13
4,2,30035,5,0,202279,3,Natural Sweetener,17,13


In [14]:
# Per-user, per-product stats from PRIOR history
user_prod_stats = (
    op_prior.groupby(['user_id', 'product_id'], as_index=False)
    .agg(
        times_bought=('order_id', 'count'),
        last_order_number=('order_number', 'max'),
        reorder_rate=('reordered', 'mean')
    )
)

# merge names for readability
user_prod_stats = user_prod_stats.merge(prod_meta, on='product_id', how='left')

def recommend_top5_frequency(user_id: int, n: int = 5):
    """
    Non-ML baseline:
    - prefer items bought often
    - prefer items bought recently
    - prefer items with high reorder_rate
    """
    df = user_prod_stats[user_prod_stats['user_id'] == user_id].copy()
    if df.empty:
        return pd.DataFrame(columns=['product_id','product_name','times_bought','reorder_rate','last_order_number'])

    df = df.sort_values(
        by=['times_bought', 'last_order_number', 'reorder_rate'],
        ascending=[False, False, False]
    )
    return df[['product_id','product_name','times_bought','reorder_rate','last_order_number']].head(n)

# try
recommend_top5_frequency(1)

,product_id,product_name,times_bought,reorder_rate,last_order_number
0,196,Soda,10,0.900000,10
3,12427,Original Beef Jerky,10,0.900000,10
1,10258,Pistachios,9,0.888889,10
8,25133,Organic String Cheese,8,0.875000,10
4,13032,Cinnamon Toast Crunch,3,0.666667,10


In [15]:
# Global popularity + reorder rate per product (from PRIOR)
global_prod_stats = (
    op_prior.groupby('product_id', as_index=False)
    .agg(
        global_times=('order_id', 'count'),
        global_reorder_rate=('reordered', 'mean')
    )
).merge(prod_meta, on='product_id', how='left')

# User aisle preferences
user_aisle_pref = (
    op_prior.groupby(['user_id', 'aisle_id'], as_index=False)
    .agg(aisle_count=('order_id', 'count'))
)

# Popular products within each aisle
aisle_prod_pop = (
    op_prior.groupby(['aisle_id', 'product_id'], as_index=False)
    .agg(aisle_times=('order_id', 'count'))
).merge(global_prod_stats, on=['product_id','aisle_id'], how='left')

def recommend_top5_aisle(user_id: int, n: int = 5, top_aisles: int = 3):
    """
    Content-based recommender:
    1) find user's top aisles
    2) recommend popular products from those aisles
       (rank by aisle popularity + global reorder rate)
    """
    # products already seen by user
    seen = set(user_prod_stats[user_prod_stats['user_id'] == user_id]['product_id'].unique())

    # top aisles for user
    topA = (
        user_aisle_pref[user_aisle_pref['user_id'] == user_id]
        .sort_values('aisle_count', ascending=False)
        .head(top_aisles)['aisle_id']
        .tolist()
    )
    if not topA:
        return pd.DataFrame(columns=['product_id','product_name','aisle_id','aisle_times','global_reorder_rate'])

    cand = aisle_prod_pop[aisle_prod_pop['aisle_id'].isin(topA)].copy()

    # recommend either: (A) reorders allowed, or (B) new items only
    # Here: NEW items only (more interesting). Comment this out if you want reorders too.
    cand = cand[~cand['product_id'].isin(seen)]

    cand = cand.sort_values(
        by=['aisle_times', 'global_reorder_rate', 'global_times'],
        ascending=[False, False, False]
    )

    return cand[['product_id','product_name','aisle_id','aisle_times','global_reorder_rate']].head(n)

# try
recommend_top5_aisle(1)

,product_id,product_name,aisle_id,aisle_times,global_reorder_rate
27077,10957,Fridge Pack Cola,77,18269,0.715255
8257,3599,Baked Aged White Cheddar Rice and Corn Puffs,23,13691,0.643635
8337,15902,100 Calorie Per Bag Popcorn,23,12822,0.678209
44079,35140,Organic Whole Cashews,117,12816,0.605025
27101,12916,Ginger Ale,77,12536,0.521219


In [16]:
# Build user -> set of products in their TRAIN order (the "next order" we are trying to predict)
op_train = order_products_train.merge(
    orders[['order_id', 'user_id']],
    on='order_id',
    how='left'
)

truth = (
    op_train.groupby('user_id')['product_id']
    .apply(lambda s: set(s.values))
    .to_dict()
)

import random

def precision_at_k(recs, truth_set, k=5):
    recs = list(recs)[:k]
    if not recs:
        return 0.0
    hits = len(set(recs) & truth_set)
    return hits / k

def recall_at_k(recs, truth_set, k=5):
    recs = list(recs)[:k]
    if not truth_set:
        return 0.0
    hits = len(set(recs) & truth_set)
    return hits / len(truth_set)

def eval_recommender(user_ids, recommender_fn, k=5):
    ps, rs = [], []
    for uid in user_ids:
        if uid not in truth:
            continue
        rec_df = recommender_fn(uid, n=k)
        recs = rec_df['product_id'].tolist() if 'product_id' in rec_df else []
        ps.append(precision_at_k(recs, truth[uid], k))
        rs.append(recall_at_k(recs, truth[uid], k))
    return (sum(ps)/len(ps), sum(rs)/len(rs)) if ps else (0.0, 0.0)

# sample users for speed
all_users = list(truth.keys())
sample_users = random.sample(all_users, k=min(2000, len(all_users)))

p1, r1 = eval_recommender(sample_users, recommend_top5_frequency, k=5)
p2, r2 = eval_recommender(sample_users, recommend_top5_aisle, k=5)

print("Baseline Frequency  P@5:", round(p1,4), " R@5:", round(r1,4))
print("Aisle Content-Based P@5:", round(p2,4), " R@5:", round(r2,4))

Baseline Frequency  P@5: 0.3527  R@5: 0.2288
Aisle Content-Based P@5: 0.0186  R@5: 0.009
